In [1]:
from pathlib import Path
import pandas as pd
import torch
import matplotlib.pyplot as plt
import xarray as xr
import yaml
import geopandas as gpd
import numpy as np
import pickle
from neuralhydrology.evaluation.metrics import calculate_all_metrics
from neuralhydrology.evaluation.metrics import calculate_metrics
from neuralhydrology.evaluation.metrics import missed_peaks
import neuralhydrology

In [2]:
# ------------------- Paths -------------------
RUN_DIR = Path("./runs")

# Ensemble metrics

In [65]:
# run_pattern = "precip_prcp_mm_day_seed_*"
# run_pattern = "precip_prcp_chirps_mm_day_seed_*"
# run_pattern = "precip_prcp_gauge_mm_day_seed_*"
# run_pattern = "precip_prcp_mswep_mm_day_seed_*"
# run_pattern = "precip_prcp_mm_day_prcp_chirps_mm_day_seed_*" 
# run_pattern = "precip_prcp_mm_day_prcp_mswep_mm_day_seed_*"
# run_pattern = "precip_prcp_mm_day_prcp_gauge_mm_day_seed_*"
# run_pattern = "precip_prcp_chirps_mm_day_prcp_mswep_mm_day_seed_*"
# run_pattern = "precip_prcp_chirps_mm_day_prcp_gauge_mm_day_seed_*"
# run_pattern = "precip_prcp_mswep_mm_day_prcp_gauge_mm_day_seed_*"
# run_pattern = "precip_prcp_mm_day_prcp_chirps_mm_day_prcp_mswep_mm_day_seed_*"
run_pattern = "precip_prcp_mm_day_prcp_chirps_mm_day_prcp_gauge_mm_day_seed_*"
# run_pattern = "precip_prcp_mm_day_prcp_mswep_mm_day_prcp_gauge_mm_day_seed_*" 
# run_pattern = "precip_prcp_chirps_mm_day_prcp_mswep_mm_day_prcp_gauge_mm_day_seed_*" 
# run_pattern = "precip_prcp_mm_day_prcp_chirps_mm_day_prcp_mswep_mm_day_prcp_gauge_mm_day_seed_*" 

matched_paths = sorted(RUN_DIR.glob(f"{run_pattern}/validation/model_epoch030/validation_results.p"))
print(f"Found {len(matched_paths)} runs: {[p.parts[-4] for p in matched_paths]}")

Found 8 runs: ['precip_prcp_mm_day_prcp_chirps_mm_day_prcp_gauge_mm_day_seed_111_2202_225728', 'precip_prcp_mm_day_prcp_chirps_mm_day_prcp_gauge_mm_day_seed_222_2202_230323', 'precip_prcp_mm_day_prcp_chirps_mm_day_prcp_gauge_mm_day_seed_333_2202_230920', 'precip_prcp_mm_day_prcp_chirps_mm_day_prcp_gauge_mm_day_seed_444_2202_231516', 'precip_prcp_mm_day_prcp_chirps_mm_day_prcp_gauge_mm_day_seed_555_2302_110041', 'precip_prcp_mm_day_prcp_chirps_mm_day_prcp_gauge_mm_day_seed_666_2302_110649', 'precip_prcp_mm_day_prcp_chirps_mm_day_prcp_gauge_mm_day_seed_777_2302_111256', 'precip_prcp_mm_day_prcp_chirps_mm_day_prcp_gauge_mm_day_seed_888_2302_111904']


In [66]:
# Load all runs
all_runs_data = []
for file_path in matched_paths:
    with open(file_path, "rb") as f:
        all_runs_data.append(pickle.load(f))

# Average the simulated flows across seeds, per basin
ensemble_data = {}

for basin_id in all_runs_data[0].keys():
    # Stack simulated flows from all seeds: shape (n_seeds, n_timesteps, time_step)
    sims = np.stack([
        run[basin_id]['1D']['xr']['QObs_mm_d_sim'].values
        for run in all_runs_data
    ], axis=0)
    
    mean_sim = np.mean(sims, axis=0)  # Average across seeds
    
    # Copy structure from first run, replace sim with ensemble mean
    xr_ensemble = all_runs_data[0][basin_id]['1D']['xr'].copy()
    xr_ensemble['QObs_mm_d_sim'].values[:] = mean_sim
    
    ensemble_data[basin_id] = {'1D': {'xr': xr_ensemble}}

# ensemble_data

In [67]:
# Now compute metrics on the ensemble mean
all_metric_names = [
    'NSE', 'MSE', 'RMSE', 'KGE', 'Alpha-NSE', 'Pearson-r',
    'Beta-KGE', 'Beta-NSE', 'FHV', 'FMS', 'FLV',
    'Peak-Timing', 'Missed-Peaks', 'Peak-MAPE'
]

all_metrics = {}
for basin_id, basin_data in ensemble_data.items():
    xr_ds = basin_data['1D']['xr'].isel(time_step=0)
    
    all_metrics[basin_id] = calculate_metrics(
        obs=xr_ds['QObs_mm_d_obs'],
        sim=xr_ds['QObs_mm_d_sim'],
        metrics=all_metric_names,
        resolution="1D",
        datetime_coord="date"
    )

df_metrics = pd.DataFrame(all_metrics).T
df_metrics.index.name = 'basin_id'

df_metrics

,NSE,MSE,RMSE,KGE,Alpha-NSE,Pearson-r,Beta-KGE,Beta-NSE,FHV,FMS,FLV,Peak-Timing,Missed-Peaks,Peak-MAPE
basin_id,,,,,,,,,,,,,,
CAMELS_UY_10,0.236483,1.081231,1.039823,0.322836,1.397177,0.843831,1.525749,0.411553,73.810684,-32.321404,16.454052,1.823529,0.743590,37.428272
CAMELS_UY_11,0.301370,1.762196,1.327477,0.637819,1.023419,0.662004,1.128004,0.079104,-7.626642,63.438129,41.650261,1.000000,0.380000,54.097702
CAMELS_UY_15,0.769336,0.901323,0.949380,0.740651,0.827153,0.880885,1.152307,0.061113,-15.608674,-24.636763,3.530278,0.500000,0.379310,34.499378
CAMELS_UY_16,0.318212,5.522952,2.350096,0.554317,0.813813,0.603023,0.920149,-0.031503,-17.779240,-11.859025,-680.757080,0.909091,0.322581,44.738697
CAMELS_UY_2,0.626174,2.295542,1.515105,0.802629,1.092469,0.833336,1.051261,0.033541,21.202534,-22.529041,51.241272,1.631579,0.800000,53.658504
CAMELS_UY_3,0.840284,1.507755,1.227907,0.915132,0.977384,0.918566,1.007719,0.004298,0.609690,-22.413557,38.742924,0.904762,0.387755,34.976673
CAMELS_UY_5,0.797009,4.407210,2.099336,0.848259,0.972627,0.896974,1.107989,0.042778,3.675534,-7.823112,-41.756500,0.812500,0.555556,34.794575
CAMELS_UY_6,0.864211,1.308228,1.143778,0.907597,0.995979,0.932437,1.062907,0.034505,0.011896,-23.002653,-129.733963,1.350000,0.564103,30.648502
CAMELS_UY_7,0.585748,7.329543,2.707313,0.717278,0.909488,0.778541,1.150649,0.056842,-8.470545,-26.543600,60.160904,1.105263,0.632653,49.538216


In [68]:
save_name = run_pattern.split("_seed_")[0]
df_metrics.to_csv(f"./all_ensemble_metrics/{save_name}.csv")